In [1]:
library(bigstatsr)
library(data.table)
library(dplyr)
library(rsvd)
library(glmnet)
library(Matrix)
library(knitr)
library(here)
library(CLAMP)

source(here("config.R"))

set.seed(config$ARCHS4$RANDOM_SVD_SEED)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: Matrix

Loaded glmnet 4.1-10

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses



In [12]:
archs4_CLAMPfull <- readRDS(here('output/archs4/archs4_CLAMP_C2CP.rds'))

In [3]:
archs4_CLAMPfull_summary <- data.frame(as.matrix(archs4_CLAMPfull$summary))

In [13]:
archs4_CLAMPfull_U <- data.frame(as.matrix(archs4_CLAMPfull$U))

In [14]:
head(archs4_CLAMPfull_U)

,LV1,LV2,LV3,LV4,LV5,LV6,LV7,LV8,LV9,LV10,⋯,LV2357,LV2358,LV2359,LV2360,LV2361,LV2362,LV2363,LV2364,LV2365,LV2366
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
C2CP_SA_B_CELL_RECEPTOR_COMPLEXES,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
C2CP_SA_CASPASE_CASCADE,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
C2CP_SA_G1_AND_S_PHASES,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
C2CP_SA_MMP_CYTOKINE_CONNECTION,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
C2CP_SA_PROGRAMMED_CELL_DEATH,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
C2CP_SA_PTEN_PATHWAY,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0


In [15]:
nrow(archs4_CLAMPfull_U)

[1] 3109

In [4]:
archs4_CLAMPfull <- NULL

In [5]:
output_dir <- config$ARCHS4$DATASET_FOLDER
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

In [6]:
# meta
meta <- readRDS(file.path(output_dir, "metadata_filtered.rds"))
n_genes_thin <- meta$n_genes_thin
n_samples <- meta$n_samples

archs4_genes <- meta$gene_symbols_thin

all_samples <- readRDS(file.path(output_dir, "all_samples.rds"))
sample_names <- all_samples[seq_len(n_samples)]

In [7]:
c2_gmt <- CLAMP:::read_gmt(here('data/archs4/c2.cp.v2026.1.Hs.symbols.gmt'))
names(c2_gmt) <- paste0("C2CP_", names(c2_gmt))

c2_pathMat <- gmtListToSparseMat(list(C2CP = c2_gmt))
c2_matched <- getMatchedPathwayMat(c2_pathMat, archs4_genes)

There are 11049 genes in the intersection between data and prior

Removing 1006 pathways



In [8]:
total_c2 <- colnames(c2_matched) %>% unique() %>% length()

In [9]:
archs4_CLAMPfull_summary <- archs4_CLAMPfull_summary %>% 
dplyr::mutate(FDR = as.numeric(FDR)) %>%
dplyr::mutate(AUC = as.numeric(AUC))

In [10]:
archs4_CLAMPfull_summary_sig <- archs4_CLAMPfull_summary %>% 
dplyr::filter(FDR < 0.1) %>% 
dplyr::filter(AUC > 0.6)

sig_lvs <- archs4_CLAMPfull_summary_sig %>% 
pull(LV) %>% 
unique() %>% 
length()

In [11]:
sig_lvs / total_c2 * 100

[1] 12.73721